# Thu thập và gán nhãn — dữ liệu, code và file đầu ra

**Nhóm Bảo, Cương, Phi, Phương — notebook 01.**

Run All **chạy thật đường xử lý offline**: đọc JSON đã thu thập → chuyển toàn bộ
comment/reply thành CSV → chạy cleaner → chạy lại luật chấp nhận nhãn từ cache
đã lưu → xuất file. Không cào Facebook mới hoặc gọi LLM/Argilla trong notebook.

Mỗi phần hiển thị **dữ liệu trước/sau, số lượng, tên file và nội dung file**.
Các khối “Xem code” đọc source từ module, không tự thực thi đoạn source đó.
Notebook 02 dùng v7 đã chốt để huấn luyện, đánh giá và demo; đây là hai luồng có
đầu vào riêng, không nhận các CSV vừa tạo ở đây là v7.


## 0. Môi trường và thư mục output

Chạy trong repo với Python 3.12 và `requirements/notebook.txt`. File nguồn thiếu
sẽ được tải Drive theo manifest/hash. Màu bảng/code được đặt cố định để đọc được
trong cả theme sáng và tối. Các file vừa sinh nằm tại `ARTIFACT_DIR` in dưới đây.


In [1]:
import os, sys
from pathlib import Path

# Mở notebook trong repo; không cần nhúng lại module bằng %%writefile.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "pipeline/__init__.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook trong thư mục project có pipeline/ và external/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import hashlib, json, subprocess
import pandas as pd
from IPython.display import HTML
from pipeline.presentation.notebook_display import (
    display_readable as display, code_excerpt_html, file_record)
from pipeline.collect.json_to_csv import flatten_posts
from pipeline.collect.snapshots import ensure_snapshots
from pipeline.prepare.clean_for_labeling import clean_facebook_comments
from pipeline.label.label_utils import join_metadata

SOURCE_ROOT = Path(os.environ.get("SPAMHAM_SOURCE_DIR", str(PROJECT_ROOT / "data")))
ARTIFACT_DIR = Path(os.environ.get("SPAMHAM_OUTPUT_DIR", str(PROJECT_ROOT / "artifacts/collection")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
created_files = []

def show_code(relative_path, function_name, max_lines=30):
    # Source được escape và chỉ hiển thị; không import/chạy crawler.
    snippet = code_excerpt_html(PROJECT_ROOT / relative_path, function_name, max_lines, title=relative_path)
    summary = '<summary style="background:#e2e8f0!important;color:#0f172a!important;padding:9px;cursor:pointer;">Xem code: '+function_name+'</summary>'
    display(HTML('<details>'+summary+snippet+'</details>'))

def show_file(path, rows, content, status="vừa ghi"):
    item = file_record(path, rows, content, root=PROJECT_ROOT, status=status)
    display(pd.DataFrame([item]))
    return item

print("Chế độ: xử lý thật từ snapshot/cache đã lưu; không gọi Facebook/LLM/Argilla.")
print("Thư mục file đầu ra:", ARTIFACT_DIR.resolve())


Chế độ: xử lý thật từ snapshot/cache đã lưu; không gọi Facebook/LLM/Argilla.
Thư mục file đầu ra: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\.work\runs\readable_collection\artifacts


## 1. Đầu vào thật: file JSON, group và bài đăng

Runner `pipeline/collect/scrape_ptit_groups.py` gọi bộ thư viện nền ở
`external/facebook_scraper/`, dựa trên repo
https://github.com/mohdtalal3/facebook_post_comment_scraper . Bảng dưới cho biết
**file nào đã có sẵn**, chứa bao nhiêu bản ghi và thuộc công đoạn nào.


In [2]:
source_names = ["groups_data_phase2.json", "comments_phase2.csv", "comments_phase2_cleaned.csv",
                "predictions_glm_cache_phase2.json", "comments_labeled_phase2.csv"]
snapshot_paths = ensure_snapshots(SOURCE_ROOT, source_names)
posts = json.loads(snapshot_paths["groups_data_phase2.json"].read_text(encoding="utf-8-sig"))
read_csv = lambda name: pd.read_csv(snapshot_paths[name], dtype=str, keep_default_na=False)
historical_flat = read_csv("comments_phase2.csv")
historical_clean = read_csv("comments_phase2_cleaned.csv")
official = read_csv("comments_labeled_phase2.csv")
cache = json.loads(snapshot_paths["predictions_glm_cache_phase2.json"].read_text(encoding="utf-8"))
sources = [("groups_data_phase2.json", len(posts), "JSON bài đăng + comments/replies, chưa nhãn"),
           ("comments_phase2.csv", len(historical_flat), "CSV đã lọc trong lịch sử, chưa nhãn"),
           ("comments_phase2_cleaned.csv", len(historical_clean), "CSV cleaned đúng thứ tự ID cache lịch sử"),
           ("predictions_glm_cache_phase2.json", len(cache), "ID → gợi ý label/confidence đã lưu"),
           ("comments_labeled_phase2.csv", len(official), "CSV nhãn chốt đã lưu, có label")]
display(pd.DataFrame([file_record(snapshot_paths[name], count, meaning, PROJECT_ROOT, "đã có sẵn; vừa đọc/kiểm hash")
                      for name, count, meaning in sources]))
groups = pd.DataFrame([{"group_id": p["group_id"], "group_name": p["group_name"]} for p in posts])
group_table = groups.groupby(["group_id", "group_name"]).size().rename("số bài trong JSON").reset_index()
group_table["link group"] = "https://www.facebook.com/groups/"+group_table.group_id.astype(str)+"/"
display(group_table)


#,file,trạng thái,số bản ghi,KB,chứa
0,data/raw/groups_data_phase2.json,đã có sẵn; vừa đọc/kiểm hash,2170,4411.2,"JSON bài đăng + comments/replies, chưa nhãn"
1,data/intermediate/comments_phase2.csv,đã có sẵn; vừa đọc/kiểm hash,9569,1700.3,"CSV đã lọc trong lịch sử, chưa nhãn"
2,data/intermediate/comments_phase2_cleaned.csv,đã có sẵn; vừa đọc/kiểm hash,9398,2195.1,CSV cleaned đúng thứ tự ID cache lịch sử
3,data/intermediate/predictions_glm_cache_phase2.json,đã có sẵn; vừa đọc/kiểm hash,6605,411.2,ID → gợi ý label/confidence đã lưu
4,data/labeled/comments_labeled_phase2.csv,đã có sẵn; vừa đọc/kiểm hash,6303,867.3,"CSV nhãn chốt đã lưu, có label"


#,group_id,group_name,số bài trong JSON,link group
0,1130813927934700,D26 (2008) PTIT - Học viện Công nghệ Bưu chính Viễn thông,500,https://www.facebook.com/groups/1130813927934700/
1,1210825580901941,PTIT Confessions - Sinh Viên Học Viện Công Nghệ Bưu Chính Viễn Thông ✅,500,https://www.facebook.com/groups/1210825580901941/
2,1605563914144178,D26 Học Viện Công Nghệ Bưu Chính Viễn Thông (2K8 PTIT),500,https://www.facebook.com/groups/1605563914144178/
3,450310163844833,Cộng đồng sinh viên PTIT,500,https://www.facebook.com/groups/450310163844833/
4,778910719727961,D21 PTIT Hà Nội,170,https://www.facebook.com/groups/778910719727961/


## 2. Bình luận/reply đã cào được nằm ở đâu? — ý 1

Cell này **đọc nội dung thật trong JSON**, hiển thị một bài, comment và reply.
Khi thu thập live, runner gọi `fetch_posts()` → `fetch_comments_for_post()` →
`fetch_comments()/fetch_replies()` rồi writer lưu JSON/JSONL và state. Trong lần
Run All này JSON là đầu vào đã có, không phải vừa cào lại trên Facebook.

Snapshot không chứng minh đã lấy hết mọi comment/reply. Giới hạn phân trang,
checkpoint và lịch sử nguồn nằm ở `docs/provenance/`.


In [3]:
example = next(p for p in posts if any(c.get("replies") for c in p.get("comments", [])))
display(pd.Series({key: example.get(key) for key in ["group_id", "group_name", "post_id", "permalink", "created_at", "scraped_at"]}).to_frame("bài thật trong JSON"))
raw_rows = pd.DataFrame(flatten_posts(posts))
example_rows = pd.DataFrame(flatten_posts([example]))
raw_examples = pd.concat([example_rows[example_rows.comment_type.eq("comment")].head(3),
                          example_rows[example_rows.comment_type.eq("reply")].head(3)])
display(raw_examples[["post_id", "comment_type", "text", "reaction_count"]])
display(raw_rows.comment_type.value_counts().rename("số dòng trong JSON"))
show_code("pipeline/collect/scrape_ptit_groups.py", "on_batch")
show_code("external/facebook_scraper/main.py", "fetch_comments_for_post")
show_code("pipeline/collect/scrape_ptit_groups.py", "save_records")


#,bài thật trong JSON
group_id,778910719727961
group_name,D21 PTIT Hà Nội
post_id,2015992686019752
permalink,https://www.facebook.com/groups/d21ptithanoii/posts/2015992686019752/
created_at,2026-03-25 17:35:43
scraped_at,2026-09-27 10:06:57


#,post_id,comment_type,text,reaction_count
0,2015992686019752,comment,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,1
1,2015992686019752,comment,"Cô Chi nhé, bao điểm cao luôn",0
2,2015992686019752,comment,Thầy hùng nhé,0
3,2015992686019752,reply,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,0


comment_type,số dòng trong JSON
comment,9438
reply,4395


## 3. Chạy thật JSON → CSV: toàn bộ snapshot

`flatten_posts()` vừa chạy trên toàn bộ JSON. Cell dưới bỏ text rỗng, ghi CSV
mới rồi **đọc lại file trên đĩa** để kiểm tra. Không lọc theo tập tham chiếu ở
nhánh này, nên không nhận file mới là bản CSV lịch sử 9.569 dòng.


In [4]:
converted = raw_rows[raw_rows.text.str.strip().ne("")].reset_index(drop=True)
csv_path = ARTIFACT_DIR / "comments_from_snapshot.csv"
converted.to_csv(csv_path, index=False, encoding="utf-8-sig")
converted_disk = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
assert converted.fillna("").astype(str).equals(converted_disk)
display(pd.DataFrame({"bước": ["comment/reply trong JSON", "bỏ text rỗng", "CSV vừa ghi"],
                      "số dòng": [len(raw_rows), len(raw_rows)-len(converted), len(converted_disk)]}))
display(converted_disk[["post_id", "comment_type", "text", "reaction_count"]].head(10))
created_files.append(show_file(csv_path, len(converted_disk), ", ".join(converted_disk.columns)))
show_code("pipeline/collect/json_to_csv.py", "flatten_posts")


#,bước,số dòng
0,comment/reply trong JSON,13833
1,bỏ text rỗng,129
2,CSV vừa ghi,13704


#,post_id,comment_type,text,reaction_count
0,2015992686019752,comment,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,1
1,2015992686019752,comment,"Cô Chi nhé, bao điểm cao luôn",0
2,2015992686019752,comment,Thầy hùng nhé,0
3,2015992686019752,reply,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,0
4,2015992686019752,comment,"Thầy Kiên nha em, thầy 10 điểm ko có nhưng",0
5,2015992686019752,comment,Cô thư điểm cao,0
6,2179604596325226,comment,Chụp ảnh ib anh tư vấn nhe,0
7,2179604596325226,reply,Nông Thành Long có đưa ảnh em thì dùng cái pts rồi nghe chưa :)))),1
8,2176185506667135,comment,"Không nhé, năm ngoái làm về AI thì điểm cao, còn nếu về web app thì khó được 8 lắm",0
9,2176185506667135,reply,"SilverPanda3602 app đã deploy lên chợ, có ci cd, ứng dụng có tích hợp AI thì sao anh, đủ 8 không ạ",0


#,file,trạng thái,số bản ghi,KB,chứa
0,.work/runs/readable_collection/artifacts/comments_from_snapshot.csv,vừa ghi,13704,2331.6,"post_id, group_id, group_name, created_at, comment_type, text, reaction_count"


## 4. Chạy thật cleaner và xem text trước/sau — ý 2

Gọi `clean_facebook_comments()` từ module, có ghi file CSV thật. Cleaner giữ
text gốc, thêm cleaned_text, khử trùng/lọc dòng theo cấu hình. Đây là clean để
chuẩn bị nhãn; tách từ/đối chứng stop words cho mô hình ở notebook 02.


In [5]:
cleaned_path = ARTIFACT_DIR / "comments_cleaned_from_snapshot.csv"
clean_facebook_comments([csv_path], cleaned_path)
cleaned_disk = pd.read_csv(cleaned_path, dtype=str, keep_default_na=False)
changed = cleaned_disk[cleaned_disk.text.ne(cleaned_disk.cleaned_text)]
display(changed[["text", "cleaned_text"]].head(10))
display(pd.DataFrame({"trạng thái": ["trước clean", "sau khử trùng/lọc", "dòng text được chuẩn hóa"],
                      "số dòng": [len(converted_disk), len(cleaned_disk), len(changed)]}))
created_files.append(show_file(cleaned_path, len(cleaned_disk), ", ".join(cleaned_disk.columns)))
show_code("pipeline/prepare/clean_for_labeling.py", "preprocess")


  - File: comments_from_snapshot.csv | thô:  13704 | giữ:  11366
  - Tổng dòng thô:             13704
  - Bỏ (dòng trùng lặp):        2338
  - Bỏ (rỗng/chỉ dấu câu):       392
  - Còn lại sau làm sạch:      10974 (80.1%)
  - Có chứa URL:                 331
  - Có chứa SĐT:                 109
  -> Đã lưu CSV: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\.work\runs\readable_collection\artifacts\comments_cleaned_from_snapshot.csv


#,text,cleaned_text
12,"IntelligentOctopus8657 AI nó về bài toán thực tế, nếu đề tài hay làm dạng nghiên cứu đưa ra giải pháp thì điểm đã cao rồi còn đề tài chung chung xử lý những thứ bình thường như chatbot thì cần tech cao, phải tối ưu hơn những cái đã có","IntelligentOctopus8657 AI nó về bài toán thực tế, nếu đề tài hay làm dạng nghiên cứu đưa ra giải pháp thì điểm đã cao rồi còn đề tài chung chung xử lý những thứ bình thường như chatbot thì cần tech cao, phải tối ưu hơn những cái đã có"
19,"RelaxingAvocado9388 tuỳ vào hội đồng nhé, ví dụ hội đồng nào có thầy hoá là y rằng thầy hỏi sâu về database","RelaxingAvocado9388 tuỳ vào hội đồng nhé, ví dụ hội đồng nào có thầy hoá là y rằng thầy hỏi sâu về database"
36,"FriendlyTurkey7620 a/chi ơi cho em hỏi, thực tập ở vnpt như nào vậy ạ","FriendlyTurkey7620 a chi ơi cho em hỏi, thực tập ở vnpt như nào vậy ạ"
41,"FriendlyTurkey7620 Anh ơi cho em hỏi với, điểm thực tập ở VNPT có được A/A+ không anh","FriendlyTurkey7620 Anh ơi cho em hỏi với, điểm thực tập ở VNPT có được A A không anh"
44,Nguyễn Thành Nam kín nhá,Nguyễn Thành Nam kín nhá
50,"e nên tìm hiểu về học phí nữa, hiện tại tuyển sinh ngành kỹ thuật đang hơn 1 triệu/tín chỉ. Nhưng quan trọng hơn là học phí năm nào cũng tăng khoảng 100k/ tín chỉ. 😅 Lúc a mới vô trường chưa tới 600k/tín mà giờ tăng gần gấp đôi rồi.","e nên tìm hiểu về học phí nữa, hiện tại tuyển sinh ngành kỹ thuật đang hơn 1 triệu tín chỉ. Nhưng quan trọng hơn là học phí năm nào cũng tăng khoảng 100k tín chỉ. Lúc a mới vô trường chưa tới 600k tín mà giờ tăng gần gấp đôi rồi."
52,Nếu phân vân về độ khó thì k nên học mấy cái này 🐧 Còn mình thấy viễn thông thì có nhiều hướng đi hơn,Nếu phân vân về độ khó thì k nên học mấy cái này Còn mình thấy viễn thông thì có nhiều hướng đi hơn
56,ElegantCorgi5380 sao em nghe a chị bảo là kì đông . cả kể ít cũng mở nhưng sẽ * 1.5 học phí,ElegantCorgi5380 sao em nghe a chị bảo là kì đông . cả kể ít cũng mở nhưng sẽ 1.5 học phí
70,"Minh Phúc địa ngục, nhưng mà chỉ cần nghe chửi , không cãi = qua môn","Minh Phúc địa ngục, nhưng mà chỉ cần nghe chửi , không cãi qua môn"
74,"Khổ thân hệ thường vãi, CLC học cực chill, ngoan thì thầy cho về sớm, đề dễ như biếu điểm luôn 😂","Khổ thân hệ thường vãi, CLC học cực chill, ngoan thì thầy cho về sớm, đề dễ như biếu điểm luôn"


#,trạng thái,số dòng
0,trước clean,13704
1,sau khử trùng/lọc,10974
2,dòng text được chuẩn hóa,2832


#,file,trạng thái,số bản ghi,KB,chứa
0,.work/runs/readable_collection/artifacts/comments_cleaned_from_snapshot.csv,vừa ghi,10974,2589.3,"post_id, group_id, group_name, created_at, comment_type, text, cleaned_text, reaction_count"


## 5. Chạy thật bước chấp nhận nhãn từ cache — offline

`ai_assisted_spam_ham.py` là công cụ gọi LLM để tạo gợi ý. Ở đây **không gọi lại
LLM**: chạy CLI `auto_accept_labels --skip-argilla` để áp ngưỡng 0,75 vào cache
đã lưu. Confidence là điểm AI đưa ra, không phải tỷ lệ đúng đã được đo.

Để giữ đúng ID, bước này dùng **CSV cleaned lịch sử 9.398 dòng đi kèm cache**,
không áp ID cache vào CSV mới ở mục 4. Hai nhánh có phạm vi/lọc khác nhau. Output
gồm bảng đủ dòng (có dòng chưa nhãn) và CSV chỉ gồm những dòng được chấp nhận.


In [6]:
accepted_all_path = ARTIFACT_DIR / "accepted_from_saved_cache.csv"
command = [sys.executable, "-m", "pipeline.label.auto_accept_labels",
           "--csv-in", str(snapshot_paths["comments_phase2_cleaned.csv"]),
           "--cache", str(snapshot_paths["predictions_glm_cache_phase2.json"]),
           "--threshold", "0.75", "--skip-argilla", "--export-csv", str(accepted_all_path)]
completed = subprocess.run(command, cwd=PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8")
print("Đã chạy module: pipeline.label.auto_accept_labels --skip-argilla --threshold 0.75")
if completed.returncode:
    raise RuntimeError(completed.stderr or completed.stdout)
print(completed.stdout)
accepted_all = pd.read_csv(accepted_all_path, dtype=str, keep_default_na=False)
accepted = accepted_all[accepted_all.official_label.isin(["ham", "spam"])].copy()
accepted_path = ARTIFACT_DIR / "labels_auto_accepted.csv"
accepted.to_csv(accepted_path, index=False, encoding="utf-8-sig")
display(accepted.official_label.value_counts().rename("nhãn vừa chấp nhận từ cache"))
display(pd.concat([accepted[accepted.official_label.eq(label)].head(3) for label in ["ham", "spam"]])[
    ["id", "text", "official_label", "ai_confidence"]])
created_files.append(show_file(accepted_all_path, len(accepted_all), "đủ dòng nguồn; official_label rỗng nếu chưa được nhận"))
created_files.append(show_file(accepted_path, len(accepted), "chỉ các ID có official_label ham/spam sau áp ngưỡng"))
show_code("pipeline/label/ai_assisted_spam_ham.py", "predict_batch_with_glm", 25)


Đã chạy module: pipeline.label.auto_accept_labels --skip-argilla --threshold 0.75
[*] Tổng số bản ghi trong cache: 6,605
[*] Ngưỡng Confidence: >= 0.75
[+] Bản ghi đủ điều kiện gán nhãn chính thức (>= 0.75): 6,302 (95.41%)
    - Spam: 1,106
    - Ham:  5,196
[-] Bản ghi cần thẩm định thủ công (< 0.75): 303 (4.59%)

[*] Đang tạo file CSV tổng hợp tại C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\.work\runs\readable_collection\artifacts\accepted_from_saved_cache.csv...
[+] Đã lưu file CSV: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\.work\runs\readable_collection\artifacts\accepted_from_saved_cache.csv (9,398 dòng)



official_label,nhãn vừa chấp nhận từ cache
ham,5196
spam,1106


#,id,text,official_label,ai_confidence
0,cmt_0,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,ham,0.95
1,cmt_1,"Cô Chi nhé, bao điểm cao luôn",ham,0.95
2,cmt_2,Thầy hùng nhé,ham,0.95
159,cmt_159,M cần hoa tốt nghiệp ib cho shop nha Bloom Bite,spam,0.9
276,cmt_276,"các e D22 app VMO đi nhé, chỉ onboard t2 đầu tuần và auto điểm 8 trở lên =))",spam,0.92
451,cmt_451,[ THANH XUÂN] tuyển dụng 1. Vị trí: Nhân viên kỹ thuật lắp đặt camera và Sinh viên làm thêm Thu nhập tb : 10.000.000 - 20.000.000 vnđ Không có kinh nghiệm sẽ được đào tạo bài bản. Thời gian: 8h – 17h30 ( Nghỉ trưa 12h – 13h30 ) T2 - T7 * Địa điểm: 30 ngõ 116 Nguyễn Xiển - Hạ Đình - Thanh Xuân - Hà Nội * Quan tâm ib trực tiếp hoặc gửi CV qua mail: cuc.matden@gmail.com Sdt/Zalo: 0985601191,spam,0.98


#,file,trạng thái,số bản ghi,KB,chứa
0,.work/runs/readable_collection/artifacts/accepted_from_saved_cache.csv,vừa ghi,9398,2387.8,đủ dòng nguồn; official_label rỗng nếu chưa được nhận


#,file,trạng thái,số bản ghi,KB,chứa
0,.work/runs/readable_collection/artifacts/labels_auto_accepted.csv,vừa ghi,6302,1666.9,chỉ các ID có official_label ham/spam sau áp ngưỡng


## 6. Xem nhãn chốt đã lưu trên cùng comment và xuất CSV

CSV export lịch sử có 6.303 nhãn, gồm một ID ngoài cache; chưa có log giải thích
ai đã chốt thêm. Cell này **nối/kiểm tra và xuất lại nhãn đã có**, không giả là vừa
gọi AI hoặc người duyệt. Chọn 4 ham/4 spam để nhìn rõ hai lớp, không dùng các ví dụ
đã chọn này làm thước đo chất lượng mô hình.


In [7]:
official_checked = join_metadata(official, historical_clean)
assert len(official_checked) == len(official)
sample = pd.concat([official_checked[official_checked.label.eq(label)].head(4) for label in ["ham", "spam"]]).copy()
positions = sample.id.str.extract(r"^cmt_(\d+)$", expand=False).astype(int)
matched = historical_clean.iloc[positions].reset_index(drop=True)
sample["post_id"] = matched.post_id.to_numpy()
raw_keys = set(zip(raw_rows.post_id.astype(str), raw_rows.comment_type, raw_rows.text))
assert all(key in raw_keys for key in zip(sample.post_id, sample.comment_type, sample.text))
sample["AI đã gợi ý"] = sample.id.map(lambda rid: cache.get(rid, {}).get("label", "không có trong cache"))
sample["confidence"] = sample.id.map(lambda rid: cache.get(rid, {}).get("confidence", ""))
display(sample[["id", "post_id", "text", "cleaned_text", "AI đã gợi ý", "confidence", "label"]])
display(official_checked.label.value_counts().rename("nhãn chốt trong export đã lưu"))
official_path = ARTIFACT_DIR / "labels_saved_official_export.csv"
official_checked.to_csv(official_path, index=False, encoding="utf-8-sig")
created_files.append(show_file(official_path, len(official_checked), ", ".join(official_checked.columns)))
show_code("pipeline/label/download_labeled_data.py", "extract_submitted_response")


#,id,post_id,text,cleaned_text,AI đã gợi ý,confidence,label
0,cmt_0,2015992686019752,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,ham,0.95,ham
1,cmt_1,2015992686019752,"Cô Chi nhé, bao điểm cao luôn","Cô Chi nhé, bao điểm cao luôn",ham,0.95,ham
2,cmt_2,2015992686019752,Thầy hùng nhé,Thầy hùng nhé,ham,0.95,ham
3,cmt_3,2015992686019752,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,ham,0.95,ham
159,cmt_159,2073247930294227,M cần hoa tốt nghiệp ib cho shop nha Bloom Bite,M cần hoa tốt nghiệp ib cho shop nha Bloom Bite,spam,0.9,spam
271,cmt_276,2027525204866500,"các e D22 app VMO đi nhé, chỉ onboard t2 đầu tuần và auto điểm 8 trở lên =))","các e D22 app VMO đi nhé, chỉ onboard t2 đầu tuần và auto điểm 8 trở lên ))",spam,0.92,spam
437,cmt_451,1855231318762557,[ THANH XUÂN] tuyển dụng 1. Vị trí: Nhân viên kỹ thuật lắp đặt camera và Sinh viên làm thêm Thu nhập tb : 10.000.000 - 20.000.000 vnđ Không có kinh nghiệm sẽ được đào tạo bài bản. Thời gian: 8h – 17h30 ( Nghỉ trưa 12h – 13h30 ) T2 - T7 * Địa điểm: 30 ngõ 116 Nguyễn Xiển - Hạ Đình - Thanh Xuân - Hà Nội * Quan tâm ib trực tiếp hoặc gửi CV qua mail: cuc.matden@gmail.com Sdt/Zalo: 0985601191,THANH XUÂN tuyển dụng 1. Vị trí: Nhân viên kỹ thuật lắp đặt camera và Sinh viên làm thêm Thu nhập tb : 10.000.000 - 20.000.000 vnđ Không có kinh nghiệm sẽ được đào tạo bài bản. Thời gian: 8h 17h30 ( Nghỉ trưa 12h 13h30 ) T2 - T7 Địa điểm: 30 ngõ 116 Nguyễn Xiển - Hạ Đình - Thanh Xuân - Hà Nội Quan tâm ib trực tiếp hoặc gửi CV qua mail: cuc.matden gmail.com Sdt Zalo: phonetoken,spam,0.98,spam
438,cmt_452,1855231318762557,mng quan tâm job thì gửi CV qua zalo 0965987629 nhaaaa🥰🥰,mng quan tâm job thì gửi CV qua zalo phonetoken nhaaaa,spam,0.98,spam


label,nhãn chốt trong export đã lưu
ham,5197
spam,1106


#,file,trạng thái,số bản ghi,KB,chứa
0,.work/runs/readable_collection/artifacts/labels_saved_official_export.csv,vừa ghi,6303,873.5,"id, label, cleaned_text, text, comment_type, reaction_count"


## 7. Tổng hợp: chạy phần nào và đã tạo file gì?

CSV ở notebook này là dữ liệu xử lý lại từ snapshot/cache, không được nhận là v7.
Notebook 02 có luồng riêng: v7 → preprocess → fit/CV → test → demo. Module gộp và
đối soát ở `tools/dataset_versions/`; nguồn/giới hạn ở `docs/provenance/`.


In [8]:
execution_steps = pd.DataFrame([
    {"bước": "show_code", "thực sự làm": "đọc AST/source rồi hiển thị", "gọi dịch vụ": "không"},
    {"bước": "đọc JSON", "thực sự làm": "đọc snapshot thật và kiểm hash", "gọi dịch vụ": "chỉ tải Drive nếu thiếu file"},
    {"bước": "JSON → CSV", "thực sự làm": "flatten toàn bộ, bỏ rỗng, ghi/đọc lại CSV", "gọi dịch vụ": "không"},
    {"bước": "clean", "thực sự làm": "gọi cleaner, khử trùng/lọc và ghi CSV", "gọi dịch vụ": "không"},
    {"bước": "chấp nhận nhãn", "thực sự làm": "chạy CLI áp ngưỡng trên cache lịch sử", "gọi dịch vụ": "không gọi LLM/Argilla"},
    {"bước": "nhãn chốt", "thực sự làm": "đối soát ID/text rồi xuất nhãn đã lưu", "gọi dịch vụ": "không"}])
display(execution_steps)
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in snapshot_paths.items()}
manifest = {"mode": "reprocess_saved_snapshot_and_cached_labels", "facebook_requests": False,
            "llm_argilla_calls": False, "raw_posts": len(posts), "groups": len(group_table),
            "raw_comment_reply_rows": len(raw_rows), "converted_rows": len(converted_disk),
            "cleaned_rows": len(cleaned_disk), "cached_candidates": len(accepted_all),
            "auto_accepted_rows": len(accepted), "official_export_rows": len(official_checked),
            "source_sha256": source_hashes, "output_files": created_files,
            "next_notebook": "02_thuc_nghiem_va_demo.ipynb"}
manifest_path = ARTIFACT_DIR / "collection_demo_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(created_files+[file_record(manifest_path, 1, "số lượng, hash và file đầu ra của lần chạy", PROJECT_ROOT)]))
print("Mở các file CSV thật tại:", ARTIFACT_DIR.resolve())


#,bước,thực sự làm,gọi dịch vụ
0,show_code,đọc AST/source rồi hiển thị,không
1,đọc JSON,đọc snapshot thật và kiểm hash,chỉ tải Drive nếu thiếu file
2,JSON → CSV,"flatten toàn bộ, bỏ rỗng, ghi/đọc lại CSV",không
3,clean,"gọi cleaner, khử trùng/lọc và ghi CSV",không
4,chấp nhận nhãn,chạy CLI áp ngưỡng trên cache lịch sử,không gọi LLM/Argilla
5,nhãn chốt,đối soát ID/text rồi xuất nhãn đã lưu,không


#,file,trạng thái,số bản ghi,KB,chứa
0,.work/runs/readable_collection/artifacts/comments_from_snapshot.csv,vừa ghi,13704,2331.6,"post_id, group_id, group_name, created_at, comment_type, text, reaction_count"
1,.work/runs/readable_collection/artifacts/comments_cleaned_from_snapshot.csv,vừa ghi,10974,2589.3,"post_id, group_id, group_name, created_at, comment_type, text, cleaned_text, reaction_count"
2,.work/runs/readable_collection/artifacts/accepted_from_saved_cache.csv,vừa ghi,9398,2387.8,đủ dòng nguồn; official_label rỗng nếu chưa được nhận
3,.work/runs/readable_collection/artifacts/labels_auto_accepted.csv,vừa ghi,6302,1666.9,chỉ các ID có official_label ham/spam sau áp ngưỡng
4,.work/runs/readable_collection/artifacts/labels_saved_official_export.csv,vừa ghi,6303,873.5,"id, label, cleaned_text, text, comment_type, reaction_count"
5,.work/runs/readable_collection/artifacts/collection_demo_manifest.json,vừa ghi,1,2.3,"số lượng, hash và file đầu ra của lần chạy"


Mở các file CSV thật tại: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\.work\runs\readable_collection\artifacts
